In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import h5py
from scipy.io import mmread
from scipy.sparse import csr_matrix
from glob import glob
import shutil
import gzip
import scipy

# Define data path

In [ ]:
dataset = "ViHD_mmu_brain_h"

In [ ]:
datadir_root = f"PATH/TO/{dataset}/"
datadir = f"{datadir_root}/raw_resolution/"
outdir_root = f"{datadir_root}/raw_resolution/h5"
file_path = f"PATH/TO/{dataset}/binned_outputs/square_008um/filtered_feature_bc_matrix.h5"

# Load Raw Data

In [ ]:
with open(f"{datadir_root}/gene-names.txt") as f:
    gene_use = f.read().splitlines()

gene_use

In [ ]:
with h5py.File(file_path, 'r') as f:
    features_group = f['/matrix/features']
    gene_ids = features_group['id'][:]
    gene_names = features_group['name'][:]
    feature_type = features_group['feature_type'][:]
    genome = features_group['genome'][:] if 'genome' in features_group else [b''] * len(gene_ids)
    
    gene_ids = [x.decode('utf-8') if isinstance(x, bytes) else x for x in gene_ids]
    gene_names = [x.decode('utf-8') if isinstance(x, bytes) else x for x in gene_names]
    feature_type = [x.decode('utf-8') if isinstance(x, bytes) else x for x in feature_type]

features_raw_data = pd.DataFrame({'id': gene_ids, 'name': gene_names, 'feature_type': feature_type, 'genome': [g.decode('utf-8') if isinstance(g, bytes) else str(g) for g in genome]})
features_raw_data = features_raw_data[features_raw_data['name'].isin(gene_use)]
features_raw_data = features_raw_data.drop_duplicates(subset=['name'], keep='first')
features_raw_data

# Generate h5 for All Four Simulations

In [ ]:
for simulation in ["fixed_spotSize", "dense", "fixed_spotDistance", "proportional"]:    
    file_names_all = glob(f"{datadir_root}/raw_resolution/matrix/{simulation}/*")
    reso_all = [Path(x).name.split("_", 1)[1] for x in file_names_all if Path(x).is_dir()]
    for reso in reso_all:
        if simulation == "fixed_spotSize":
            reso_name = "spotDistance"
            cnts_f = f"{datadir_root}/raw_resolution/matrix/{simulation}/{reso_name}_{reso}/cnts_fixed_spotSize_56um.tsv"
            locs_f = f"{datadir_root}/raw_resolution/matrix/{simulation}/{reso_name}_{reso}/locs_fixed_spotSize_56um.tsv"
        else:
            reso_name = "spotSize"
            cnts_f = f"{datadir_root}/raw_resolution/matrix/{simulation}/{reso_name}_{reso}/cnts_{simulation}_{reso}um.tsv"
            locs_f = f"{datadir_root}/raw_resolution/matrix/{simulation}/{reso_name}_{reso}/locs_{simulation}_{reso}um.tsv"
        outdir = outdir_root + "/" + simulation + "/" + reso_name + "_" + reso
        os.makedirs(outdir + "/spatial", exist_ok=True)
# load cnts
        exp_data = pd.read_csv(
            cnts_f,
            sep='\t',
            header=0,
            index_col=0
        )
        exp_data.index.name = None
        cols_to_keep = [gene for gene in gene_use if gene in exp_data.columns]
        exp_data = exp_data[cols_to_keep]
# load locs
        locs = pd.read_csv(
            locs_f,
            sep='\t',
            header=0,
            index_col=0
        )
        locs['in_tissue'] = 1
        locs['array_row'] = locs['y']
        locs['array_col'] = locs['x']
        x = list(locs['x'].values)
        y = list(locs['y'].values)
        x_keys_sort = sorted(set(x))
        y_keys_sort = sorted(set(y))
        x_rank_dict = {value: rank for rank, value in enumerate(x_keys_sort)}
        y_rank_dict = {value: rank for rank, value in enumerate(y_keys_sort)}
        locs = locs.replace({"array_row": y_rank_dict, "array_col": x_rank_dict})
        locs["array_row"] = locs["array_row"].astype(int)
        locs["array_col"] = locs["array_col"].astype(int)
        locs.rename(columns={'x':'pxl_col_in_fullres','y':'pxl_row_in_fullres'},inplace = True)
        locs = locs[['in_tissue','array_row','array_col','pxl_row_in_fullres','pxl_col_in_fullres']]
        locs[['pxl_col_in_fullres', 'pxl_row_in_fullres']] = locs[['pxl_col_in_fullres', 'pxl_row_in_fullres']].round().astype(int)
        locs.index.name = None
        locs.to_csv(
            f'{outdir}/spatial/tissue_positions_list.csv', 
            sep=',', 
            index=True,
            header=False
        )
        locs.to_csv(
            f"{outdir}/spatial/tissue_positions_list_with_header.csv", 
            sep=',', 
            index=True,
            header=True
        )
# generate files
        features_use = pd.DataFrame(exp_data.columns)
        features_use.columns = ['name']
        features = pd.merge(features_use, features_raw_data, left_on="name", right_on="name", how="left")
        features = features.fillna({
            'id': features['name'], 
            'feature_type': 'Gene Expression'
        })

        features['genome'] = features['genome'].fillna('')
        features = features[["id", "name", "feature_type","genome"]]
        features_columns = features.columns.values
## save matrix.mtx.gz
        reAnno_count_sparse_mtx = scipy.sparse.csr_matrix(exp_data.T.values)
        with open(os.path.join(outdir, 'matrix.mtx'), "wb") as matrix_file:
            scipy.io.mmwrite(matrix_file, reAnno_count_sparse_mtx)
        with open(os.path.join(outdir,'matrix.mtx'),'rb') as mtx_in:
            with gzip.open(os.path.join(outdir,'matrix.mtx') + '.gz','wb') as mtx_gz:
                shutil.copyfileobj(mtx_in, mtx_gz)
        os.remove(os.path.join(outdir,'matrix.mtx'))
## save barcodes.tsv.gz
        barcodesFile = pd.DataFrame(exp_data.index)
        barcodesFile.to_csv(os.path.join(outdir,"barcodes.tsv.gz"),sep='\t',header =False,index=False)
        ##save features.tsv.gz
        features.to_csv(os.path.join(outdir,"features.tsv.gz"),sep='\t',header =False,index=False)
# generate h5
        expression_data = mmread(f"{outdir}/matrix.mtx.gz").tocsc()
        features = pd.read_csv(f"{outdir}/features.tsv.gz", header=None, sep="\t")
        features.columns = ["id", "name", "feature_type","genome"]
        barcodes = pd.read_csv(f"{outdir}/barcodes.tsv.gz", header=None)
        barcodes.columns = ["barcode"]
        barcode_list = barcodes["barcode"].values
        gene_id_list = features["id"].values
        gene_name_list = features["name"].values
        gene_type_list = features["feature_type"].values
        gene_genome_list = features["genome"]
    # generate H5 file
        with h5py.File(f"{outdir}/filtered_feature_bc_matrix.h5", 'w') as f:
    # Add library_ids to attrs
            f.attrs['library_ids'] = np.array([dataset], dtype='S')
    # Add Matrix group
            group = f.create_group('matrix')
    ## Add data
            group.create_dataset('data', data=expression_data.data)
            group.create_dataset('indices', data=expression_data.indices)
            group.create_dataset('indptr', data=expression_data.indptr)
            group.create_dataset('shape', data=expression_data.shape)
            group.create_dataset('barcodes', data=np.array(barcode_list, dtype='S'))
## Add feature group
            feature_group = group.create_group('features')
### Add data
            feature_group.create_dataset('id', data=np.array(gene_id_list, dtype='S'))
            feature_group.create_dataset('name', data=np.array(gene_name_list, dtype='S'))
            feature_group.create_dataset('feature_type', data=np.array(gene_type_list, dtype='S'))
            feature_group.create_dataset('genome', data=np.array(gene_genome_list, dtype='S'))
            feature_group.create_dataset('_all_tag_keys', data=np.array([b'genome'], dtype='S'))
### Add target group
            target_group = feature_group.create_group('target_sets')
            target_group.create_dataset('AddOn Panel', data=np.arange(len(features), dtype=np.int64))
    